# 03 - Root Cause Analysis Examples

This notebook demonstrates the root cause analysis capabilities with concrete examples.

## Contents
1. Load anomalies and RCA data
2. Example 1: Revenue drop analysis
3. Example 2: Price spike analysis
4. Example 3: Zero sales rate analysis
5. Visualization of driver contributions

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import DATA_PROCESSED, ANOMALIES_FILE, RCA_DRIVERS_FILE, METRIC_FACTS_FILE, OUTPUTS_FIGURES
from src.utils import format_number, format_percentage

## 1. Load Data

In [ ]:
# Load anomalies
anomalies = pd.read_parquet(DATA_PROCESSED / ANOMALIES_FILE)
print(f"Loaded {len(anomalies)} anomaly events")

# Load RCA drivers
rca_path = DATA_PROCESSED / RCA_DRIVERS_FILE
if rca_path.exists():
    rca_drivers = pd.read_parquet(rca_path)
    print(f"Loaded {len(rca_drivers)} RCA driver records")
else:
    rca_drivers = pd.DataFrame()
    print("RCA drivers not found. Run: python -m src.root_cause")

# Load metric facts for context
metric_facts = pd.read_parquet(DATA_PROCESSED / METRIC_FACTS_FILE)
metric_facts['date'] = pd.to_datetime(metric_facts['date'])

In [ ]:
# Display anomaly summary
print("\n=== Anomaly Summary ===")
print(f"\nBy Metric:")
print(anomalies.groupby('metric_name').size())
print(f"\nBy Level:")
print(anomalies.groupby('level').size())
print(f"\nBy Direction:")
print(anomalies.groupby('direction').size())

## 2. Example 1: Revenue Drop Analysis

In [ ]:
# Find a revenue drop event
revenue_drops = anomalies[
    (anomalies['metric_name'] == 'revenue') & 
    (anomalies['direction'] == 'drop')
].sort_values('z_severity_peak', ascending=False)

if len(revenue_drops) > 0:
    example1 = revenue_drops.iloc[0]
    print("=== Example 1: Revenue Drop ===")
    print(f"Event ID: {example1['event_id']}")
    print(f"Level: {example1['level']}")
    print(f"Segment: {example1['segment_key']}")
    print(f"Peak Date: {example1['peak_date']}")
    print(f"Severity: {example1['z_severity_peak']:.1f}")
    print(f"Change: {example1['pct_change_peak']*100:.1f}%")
else:
    print("No revenue drop events found.")
    example1 = None

In [ ]:
# Get RCA drivers for this event
if example1 is not None and len(rca_drivers) > 0:
    event_drivers = rca_drivers[rca_drivers['event_id'] == example1['event_id']].sort_values('rank')
    
    if len(event_drivers) > 0:
        print(f"\nTop 5 Drivers:")
        display_cols = ['rank', 'child_segment_key', 'delta_value', 'delta_share', 'driver_score']
        display_cols = [c for c in display_cols if c in event_drivers.columns]
        print(event_drivers[display_cols].head())
        
        # Plot driver scores
        top_drivers = event_drivers.head(10)
        fig = px.bar(
            top_drivers,
            x='driver_score',
            y='child_segment_key',
            orientation='h',
            title=f'Revenue Drop Drivers - Event {example1["event_id"]}'
        )
        fig.update_layout(yaxis={'autorange': 'reversed'})
        fig.show()
    else:
        print("No drivers found for this event.")

## 3. Example 2: Price Spike Analysis

In [ ]:
# Find a price spike event
price_spikes = anomalies[
    (anomalies['metric_name'] == 'avg_price') & 
    (anomalies['direction'] == 'spike')
].sort_values('z_severity_peak', ascending=False)

if len(price_spikes) > 0:
    example2 = price_spikes.iloc[0]
    print("=== Example 2: Price Spike ===")
    print(f"Event ID: {example2['event_id']}")
    print(f"Level: {example2['level']}")
    print(f"Segment: {example2['segment_key']}")
    print(f"Peak Date: {example2['peak_date']}")
    print(f"Severity: {example2['z_severity_peak']:.1f}")
    print(f"Change: {example2['pct_change_peak']*100:.1f}%")
else:
    print("No price spike events found.")
    example2 = None

In [ ]:
# Get RCA drivers for price spike
if example2 is not None and len(rca_drivers) > 0:
    event_drivers = rca_drivers[rca_drivers['event_id'] == example2['event_id']].sort_values('rank')
    
    if len(event_drivers) > 0:
        print(f"\nTop 5 Drivers:")
        display_cols = ['rank', 'child_segment_key', 'delta_value', 'delta_share', 'driver_score']
        display_cols = [c for c in display_cols if c in event_drivers.columns]
        print(event_drivers[display_cols].head())
        
        # Check for price decomposition hints
        if 'price_decomp_hint' in event_drivers.columns:
            print("\nPrice Decomposition Hints:")
            for _, row in event_drivers.head(3).iterrows():
                hint = row.get('price_decomp_hint', 'N/A')
                print(f"  {row['child_segment_key']}: Primary driver = {hint}")
        
        # Plot
        top_drivers = event_drivers.head(10)
        fig = px.bar(
            top_drivers,
            x='driver_score',
            y='child_segment_key',
            orientation='h',
            title=f'Price Spike Drivers - Event {example2["event_id"]}'
        )
        fig.update_layout(yaxis={'autorange': 'reversed'})
        fig.show()

## 4. Example 3: Zero Sales Rate Analysis

In [ ]:
# Find a zero_sales_rate spike (potential stockout)
stockout_events = anomalies[
    (anomalies['metric_name'] == 'zero_sales_rate') & 
    (anomalies['direction'] == 'spike')
].sort_values('z_severity_peak', ascending=False)

if len(stockout_events) > 0:
    example3 = stockout_events.iloc[0]
    print("=== Example 3: Zero Sales Rate Spike (Potential Stockout) ===")
    print(f"Event ID: {example3['event_id']}")
    print(f"Level: {example3['level']}")
    print(f"Segment: {example3['segment_key']}")
    print(f"Peak Date: {example3['peak_date']}")
    print(f"Severity: {example3['z_severity_peak']:.1f}")
    print(f"Change: {example3['pct_change_peak']*100:.1f}%")
else:
    print("No zero_sales_rate spike events found.")
    example3 = None

In [ ]:
# Get RCA drivers for stockout
if example3 is not None and len(rca_drivers) > 0:
    event_drivers = rca_drivers[rca_drivers['event_id'] == example3['event_id']].sort_values('rank')
    
    if len(event_drivers) > 0:
        print(f"\nTop 5 Drivers (segments with highest zero sales increase):")
        display_cols = ['rank', 'child_segment_key', 'child_before', 'child_after', 'delta_value']
        display_cols = [c for c in display_cols if c in event_drivers.columns]
        print(event_drivers[display_cols].head())
        
        # Plot
        top_drivers = event_drivers.head(10)
        fig = px.bar(
            top_drivers,
            x='driver_score',
            y='child_segment_key',
            orientation='h',
            title=f'Zero Sales Rate Drivers - Event {example3["event_id"]}'
        )
        fig.update_layout(yaxis={'autorange': 'reversed'})
        fig.show()

## 5. Visualization: Share Changes

In [ ]:
# Visualize share changes for the first example
if example1 is not None and len(rca_drivers) > 0:
    event_drivers = rca_drivers[rca_drivers['event_id'] == example1['event_id']].sort_values('rank').head(10)
    
    if len(event_drivers) > 0 and 'share_before' in event_drivers.columns:
        # Create before/after share comparison
        fig = go.Figure()
        
        fig.add_trace(go.Bar(
            name='Before',
            x=event_drivers['child_segment_key'],
            y=event_drivers['share_before'],
            marker_color='lightblue'
        ))
        
        fig.add_trace(go.Bar(
            name='After',
            x=event_drivers['child_segment_key'],
            y=event_drivers['share_after'],
            marker_color='darkblue'
        ))
        
        fig.update_layout(
            title='Share Comparison: Before vs After Anomaly',
            barmode='group',
            xaxis_title='Segment',
            yaxis_title='Share of Total',
            yaxis_tickformat='.0%'
        )
        fig.show()
        
        # Save
        OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)
        fig.write_html(OUTPUTS_FIGURES / 'rca_share_comparison.html')

In [ ]:
# Summary of RCA insights
print("\n=== RCA Summary ===")
if len(rca_drivers) > 0:
    print(f"\nTotal driver records: {len(rca_drivers)}")
    print(f"Events with RCA: {rca_drivers['event_id'].nunique()}")
    
    # Average drivers per event
    drivers_per_event = rca_drivers.groupby('event_id').size().mean()
    print(f"Average drivers per event: {drivers_per_event:.1f}")
    
    # Distribution of child levels
    print(f"\nDriver levels:")
    print(rca_drivers['child_level'].value_counts())
else:
    print("No RCA data available.")

## Conclusion

This notebook demonstrated the root cause analysis capabilities:

1. **Revenue drops** can be traced to specific stores or departments
2. **Price spikes** include decomposition hints (revenue vs units driven)
3. **Zero sales rate spikes** identify potential stockout locations

The hierarchical RCA enables drill-down from global to state to store to department level,
providing actionable insights for business decision-making.